# TeleRAG: RAG Generation Evaluation (v4)

**What this does:** Evaluates our fine-tuned model + RAG context on TeleQnA questions.

**Key fixes in v4:**
- Correct Llama-3 chat prompt template
- Robust answer extraction supporting A-E and all model output formats
- Real 3GPP context from 35K chunks (not boilerplate)
- Loads LoRA adapter directly with Unsloth
- Tests on 10 first (sanity), then configurable for 50+

**Prerequisites:**
- Upload `kaggle_eval_input.json` as a Kaggle dataset
- Have your LoRA adapter on HuggingFace

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth transformers datasets trl accelerate bitsandbytes xformers

In [ ]:
# Cell 2: Load Model + LoRA Directly (Unsloth native)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import re
import torch
from unsloth import FastLanguageModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

max_seq_length = 4096
load_in_4bit = True

# ===== CONFIGURATION =====
USE_LORA = True
ADAPTER_REPO = "Imchaitanya/TeleRAG_LoRA"   # Your adapter repo
EVAL_DATA_PATH = "/kaggle/input/datasets/chaitanyakadupukutla/golden-rag-eval/kaggle_eval_golden_final.json"  # CHANGE

# Login (if private)
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
except:
    hf_token = None

# Load the adapter directly (Unsloth handles base model automatically)
if USE_LORA:
    print(f"Loading model with LoRA adapter from: {ADAPTER_REPO}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_REPO,          # Unsloth auto-fetches base model from adapter's config
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
        token=hf_token,
    )
else:
    print("Loading base model only")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="AliMaatouk/LLama-3-8B-Tele-it",
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
        token=hf_token,
    )

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = model.to("cuda")
model.eval()
FastLanguageModel.for_inference(model)

# Quick test: generate a simple response to verify model works
test_input = tokenizer("Hello", return_tensors="pt").to("cuda")
with torch.no_grad():
    test_out = model.generate(**test_input, max_new_tokens=20)
print("Test generation:", tokenizer.decode(test_out[0], skip_special_tokens=True))

print("Model loaded successfully.")

In [ ]:
# Cell 3: Prompt Template – EXACT match with fine-tuning
def format_options(options):
    """Convert options list to format: A) ... B) ... etc."""
    lines = []
    for i, opt in enumerate(options):
        letter = chr(65 + i)  # A, B, C, D, E...
        lines.append(f"{letter}) {opt}")
    return "\n".join(lines)

def format_prompt(item, max_context_chars=3000):
    """
    Replicates the exact prompt used during fine-tuning.
    Training input was: "### Question:\n{question_text}\n\n### Answer:\n"
    where question_text = "Question: ...\n\nOptions:\nA) ...\nB) ...\n"
    """
    # Build the question part exactly as in training
    question_text = f"Question: {item['question']}\n\nOptions:\n{format_options(item['options'])}"
    
    # Optional RAG context (if relevant and not boilerplate)
    context = item.get('context', '')
    if context and len(context) > 100 and not any(x in context for x in ['References', 'Definitions, symbols']):
        # Truncate context to fit within max_seq_length (question + answer ~ 600 tokens)
        context_block = f"\n\nRelevant information:\n{context[:max_context_chars]}"
        question_text += context_block
    
    return f"### Question:\n{question_text}\n\n### Answer:\n"

def extract_answer_letter(response_text, num_options):
    """Extract letter from model output. Trained to output 'The answer is: A'."""
    text = response_text.strip()
    max_letter = chr(64 + num_options)
    
    # Pattern 1: "The answer is: X"
    match = re.search(r'The answer is:\s*([A-Z])', text, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        if 'A' <= letter <= max_letter:
            return letter
    
    # Pattern 2: Just a letter (maybe at beginning)
    match = re.search(rf'\b([A-{max_letter}])\b', text[:50], re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # If nothing found, return None (to detect failures)
    return None

In [ ]:
# Cell 4: Load Evaluation Data
import json

with open(EVAL_DATA_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data)} questions.\n")

# Show sample prompt
sample = eval_data[0]
print("Example prompt (first 600 chars):")
print(format_prompt(sample)[:600])
print("\n...")
print(f"Ground truth answer: {sample['answer']}")

In [ ]:
# Cell 5: Run Evaluation with Fixed Generation
from tqdm import tqdm
import gc

device = "cuda"
correct = 0
results = []

for idx, item in enumerate(tqdm(eval_data, desc="Evaluating")):
    prompt = format_prompt(item)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3800).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,                  # Enough for "The answer is: X"
            do_sample=False,
            repetition_penalty=1.2,             # Discourage repetition
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    
    # Improved extraction: find "The answer is: X"
    match = re.search(r'The answer is:\s*([A-E])', response, re.IGNORECASE)
    if match:
        predicted = match.group(1).upper()
    else:
        # Fallback: first standalone letter A-E
        fallback = re.search(r'\b([A-E])\b', response[:50], re.IGNORECASE)
        predicted = fallback.group(1).upper() if fallback else '?'
    
    actual = item["answer"].strip().upper()
    is_correct = (predicted == actual)
    if is_correct:
        correct += 1
    
    results.append({
        "question": item["question"],
        "predicted": predicted,
        "actual": actual,
        "correct": is_correct,
        "raw_response": response
    })
    
    # Print first 10 for debugging
    if idx < 10:
        print(f"\n--- Sample {idx+1} ---")
        print(f"Q: {item['question'][:80]}...")
        print(f"Raw: '{response}'")
        print(f"Pred: {predicted} | Actual: {actual} -> {'✓' if is_correct else '✗'}")
    
    if idx % 50 == 0:
        torch.cuda.empty_cache()
        gc.collect()

accuracy = correct / len(eval_data) * 100
print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Total samples: {len(eval_data)}")
print(f"Correct:       {correct}")
print(f"Accuracy:      {accuracy:.2f}%")
print(f"{'='*50}")

failed = sum(1 for r in results if r['predicted'] == '?')
print(f"Failed to extract answer: {failed}")

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved.")